Processamento de Imagens Coloridas - Códigos em Python Baseado no material da disciplina PDSI

Autor: edvonaldohoracio@gmail.com / https://github.com/NaldoHoracio

Este script contém implementações em Python voltando à Análise de Qualidade de Imagem (IQA)

In [ ]:
# Download do OpenCV
!pip install opencv-python

In [ ]:
# Importação das bibliotecas
import os
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
# Define o backend da biblioteca Matplotlib para o modo Agg (Anti-Grain Geometry).
# Esse backend gera gráficos exclusivamente em memória e em arquivos de imagem, sem abrir nenhuma janela na tela.
# matplotlib.use('Agg')
#%matplotlib# Reativa a exibição de imagens no google colab

In [ ]:
# Download do repositório
if os.path.isdir("pdsi"):
  print("Diretório já existe!")
else:
  !git clone https://github.com/NaldoHoracio/pdsi

# Preparação

In [ ]:
# Bibliotecas extras: scikit-image (métricas de referência), SciPy (convolução) e pandas (tabelas)
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
from scipy.signal import fftconvolve
import pandas as pd

def ler_cinza(nome):
    # Lê a imagem do repositório em tons de cinza
    return cv2.imread(f'pdsi/imagens/{nome}', cv2.IMREAD_GRAYSCALE)

def mostrar(imagens, titulos, colunas=3, tamanho=4):
    # Exibe uma lista de imagens lado a lado
    linhas = int(np.ceil(len(imagens) / colunas))
    plt.figure(figsize=(tamanho * colunas, tamanho * linhas))
    for k, (im, t) in enumerate(zip(imagens, titulos)):
        plt.subplot(linhas, colunas, k + 1); plt.imshow(im, cmap='gray', vmin=0, vmax=255); plt.title(t, fontsize=10); plt.axis('off')
    plt.tight_layout(); plt.show()

## Imagem de referência e distorções

As métricas *Full-Reference* comparam uma imagem distorcida com a **referência** (original sem distorção).

In [ ]:
np.random.seed(0)# Semente fixada para um ruído reprodutível

ref = ler_cinza('baboon.png')

def ruido_gaussiano(img, sigma):
    return np.clip(img + np.random.normal(0, sigma, img.shape), 0, 255).astype(np.uint8)

def borramento(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigma)# kernel calculado a partir de sigma

def compressao_jpeg(img, qualidade):
    _, buffer = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, qualidade])
    return cv2.imdecode(buffer, cv2.IMREAD_GRAYSCALE)# codifica e decodifica em memória

def brilho(img, delta):
    return np.clip(img.astype(np.float64) + delta, 0, 255).astype(np.uint8)

def contraste(img, fator):
    media = img.mean()
    return np.clip(media + fator * (img - media), 0, 255).astype(np.uint8)# estica em torno da média

distorcidas = {
    'Ruído gaussiano (σ=20)': ruido_gaussiano(ref, 20),
    'Borramento (σ=2)': borramento(ref, 2),
    'JPEG (qualidade 10)': compressao_jpeg(ref, 10),
    'Brilho (+30)': brilho(ref, 30),
    'Contraste (×1.3)': contraste(ref, 1.3),
}
mostrar([ref] + list(distorcidas.values()), ['Referência'] + list(distorcidas.keys()))

# Métricas *pixel wise*

Comparam cada pixel com o pixel correspondente da outra imagem, sem considerar a vizinhança.

## MAE – Erro Médio Absoluto

$$MAE = \frac{1}{MN}\sum_{i=0}^{M-1}\sum_{j=0}^{N-1} |y(i,j) - x(i,j)|$$

onde y(i,j) e x(i,j) contém são as intensidades dos pixels das posições i, j das imagens real e distorcida, respectivamente.

Quanto **menor**, melhor (0 = imagens idênticas).

In [ ]:
def mae(y, x):
    return np.mean(np.abs(y.astype(np.float64) - x.astype(np.float64)))# float evita overflow do uint8

for nome, x in distorcidas.items():
    print(f'{nome:25s} MAE = {mae(ref, x):7.2f}')

## MSE – Erro Médio Quadrático

$$MSE = \frac{1}{MN}\sum_{i=0}^{M-1}\sum_{j=0}^{N-1} [y(i,j) - x(i,j)]^2$$

onde y(i,j) e x(i,j) contém são as intensidades dos pixels das posições i, j das imagens real e distorcida, respectivamente.

O quadrado penaliza mais os erros grandes. Quanto **menor**, melhor.

In [ ]:
def mse(y, x):
    return np.mean((y.astype(np.float64) - x.astype(np.float64)) ** 2)

for nome, x in distorcidas.items():
    print(f'{nome:25s} MSE = {mse(ref, x):8.2f} | scikit-image = {mean_squared_error(ref, x):8.2f}')

## PSNR – Relação Sinal-Ruído de Pico

$$PSNR = 10\log_{10}\left(\frac{MAX^2}{MSE}\right) \; \text{dB}, \quad MAX = 255 \text{ para 8 bits}$$

Quanto **maior**, melhor. Valores acima de ~30 dB costumam indicar boa qualidade.

In [ ]:
def psnr(y, x, max_valor=255.0):
    erro = mse(y, x)
    return np.inf if erro == 0 else 10 * np.log10(max_valor ** 2 / erro)# imagens idênticas: PSNR infinito

for nome, x in distorcidas.items():
    print(f'{nome:25s} PSNR = {psnr(ref, x):6.2f} dB | scikit-image = {peak_signal_noise_ratio(ref, x, data_range=255):6.2f} dB')

# Métricas não *pixel wise*

## SSIM – Índice de Similaridade Estrutural

Compara **luminância**, **contraste** e **estrutura** em janelas locais (gaussiana, σ = 1.5):

$$SSIM(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}, \quad C_1 = (0.01L)^2,\; C_2 = (0.03L)^2$$

onde L é sim a faixa dinâmica dos valores dos pixels, ou seja, a diferença entre o maior e o menor valor possível. Ele depende do número de bits por canal pela relação:

$$L = 2^{b} - 1$$

onde b é o número de bits por canal. Para imagens de 8 bits, L = 2⁸ − 1 = 255. Para 16 bits, L = 65535.

Se a imagem estiver normalizada em [0, 1], usa-se L = 1.

Varia até 1 (imagens idênticas). Quanto **mais próximo de 1**, melhor.

In [ ]:
def ssim(y, x, L=255, sigma=1.5):
    y, x = y.astype(np.float64), x.astype(np.float64)
    C1, C2 = (0.01 * L) ** 2, (0.03 * L) ** 2                           # constantes que evitam divisão por zero
    media = lambda im: cv2.GaussianBlur(im, (11, 11), sigma)            # média local ponderada (janela 11x11)
    mu_y, mu_x = media(y), media(x)
    var_y = media(y * y) - mu_y ** 2                                    # variância local: E[f²] − E[f]²
    var_x = media(x * x) - mu_x ** 2
    cov = media(y * x) - mu_y * mu_x                                    # covariância local entre f e g
    mapa = ((2 * mu_y * mu_x + C1) * (2 * cov + C2)) / ((mu_y ** 2 + mu_x ** 2 + C1) * (var_y + var_x + C2))
    return mapa.mean(), mapa                                            # SSIM global = média do mapa local

for nome, x in distorcidas.items():
    valor_skimage = structural_similarity(ref, x, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
    print(f'{nome:25s} SSIM = {ssim(ref, x)[0]:.4f} | scikit-image = {valor_skimage:.4f}')
# Pequenas diferenças vêm do tamanho da janela: aqui 11x11, no scikit-image ±3.5σ

### Mapa SSIM: onde a imagem foi degradada

Regiões escuras no mapa indicam baixa similaridade estrutural.

In [ ]:
plt.figure(figsize=(4 * len(distorcidas), 4))
for k, (nome, x) in enumerate(distorcidas.items()):
    valor, mapa = ssim(ref, x)
    plt.subplot(1, len(distorcidas), k + 1); plt.imshow(mapa, cmap='gray', vmin=0, vmax=1)
    plt.title(f'{nome}\nSSIM = {valor:.3f}', fontsize=10); plt.axis('off')
plt.tight_layout(); plt.show()

## Por que só o MSE não basta?

Experimento clássico: ajustamos cada distorção para ter **o mesmo MSE**. Para o PSNR elas são todas iguais, mas visualmente (e para o SSIM) a qualidade é bem diferente.

In [ ]:
ALVO_MSE = 225                                                          # equivale a um erro RMS de 15 níveis

def ajustar(distorcer, lo, hi, alvo=ALVO_MSE, passos=30):
    # Busca binária no parâmetro da distorção até atingir o MSE alvo (MSE cresce com o parâmetro)
    for _ in range(passos):
        meio = (lo + hi) / 2
        np.random.seed(0)                                               # mesmo ruído em todas as tentativas
        lo, hi = (meio, hi) if mse(ref, distorcer(meio)) < alvo else (lo, meio)
    candidatas = []
    for p in (lo, hi):                                                  # devolve o extremo com MSE mais próximo do alvo
        np.random.seed(0)
        candidatas.append(distorcer(p))
    return min(candidatas, key=lambda g: abs(mse(ref, g) - alvo))

mesmo_mse = {
    'Ruído gaussiano': ajustar(lambda s: ruido_gaussiano(ref, s), 0, 60),
    'Borramento': ajustar(lambda s: borramento(ref, s), 0.1, 10),
    'Brilho': ajustar(lambda d: brilho(ref, d), 0, 60),
    'Contraste': ajustar(lambda a: contraste(ref, 1 + a), 0, 2),
}
qualidades = range(1, 101)                                              # JPEG: qualidade é inteira, escolhe a mais próxima
mesmo_mse['JPEG'] = min((compressao_jpeg(ref, q) for q in qualidades), key=lambda g: abs(mse(ref, g) - ALVO_MSE))

titulos = [f'{n}\nMSE={mse(ref, g):.0f} | PSNR={psnr(ref, g):.1f} dB | SSIM={ssim(ref, g)[0]:.3f}' for n, g in mesmo_mse.items()]
mostrar([ref] + list(mesmo_mse.values()), ['Referência'] + titulos)

## VIF e IFC – Fidelidade da Informação Visual

Baseadas em **teoria da informação**: modelam a imagem natural como um sinal que passa por um "canal de distorção" e pelo sistema visual humano (HVS), e medem quanta informação da referência ainda chega ao observador.

- **IFC** (*Information Fidelity Criterion*): informação mútua entre referência e distorcida. Não é normalizada (vai de 0 a ∞).
- **VIF** (*Visual Information Fidelity*): IFC dividida pela informação da própria referência. VIF = 1 para imagens idênticas, < 1 com perda de informação e pode ser **> 1** quando a imagem é realçada (ex.: aumento de contraste).

> Aqui usamos a versão **no domínio do pixel** (VIF-P), com 4 escalas gaussianas. A formulação original de Sheikh e Bovik usa uma decomposição em *wavelets*, mas o comportamento é semelhante.

In [ ]:
def janela_gaussiana(N, sigma):
    eixo = np.arange(N) - (N - 1) / 2
    w = np.exp(-(eixo[:, None] ** 2 + eixo[None, :] ** 2) / (2 * sigma ** 2))
    return w / w.sum()                                                  # janela 2D normalizada (soma 1)

def vif_ifc(f, g, sigma_n2=2.0, eps=1e-10):
    f, g = f.astype(np.float64), g.astype(np.float64)
    numerador, denominador = 0.0, 0.0
    for escala in range(1, 5):                                          # 4 escalas, da mais fina para a mais grossa
        N = 2 ** (4 - escala + 1) + 1
        w = janela_gaussiana(N, N / 5)
        if escala > 1:                                                  # filtra e subamostra por 2 a cada nova escala
            f = fftconvolve(f, w, mode='valid')[::2, ::2]
            g = fftconvolve(g, w, mode='valid')[::2, ::2]
        mu_f, mu_g = fftconvolve(f, w, mode='valid'), fftconvolve(g, w, mode='valid')
        var_f = np.maximum(fftconvolve(f * f, w, mode='valid') - mu_f ** 2, 0)
        var_g = np.maximum(fftconvolve(g * g, w, mode='valid') - mu_g ** 2, 0)
        cov = fftconvolve(f * g, w, mode='valid') - mu_f * mu_g

        ganho = cov / (var_f + eps)                                     # g = ganho do canal de distorção (atenuação)
        var_v = var_g - ganho * cov                                     # σv² = ruído aditivo do canal
        ganho[var_f < eps], var_v[var_f < eps] = 0, var_g[var_f < eps]  # regiões planas da referência
        var_f[var_f < eps] = 0
        ganho[var_g < eps], var_v[var_g < eps] = 0, 0                   # regiões planas da distorcida
        var_v[ganho < 0], ganho[ganho < 0] = var_g[ganho < 0], 0        # ganho negativo não tem sentido físico
        var_v = np.maximum(var_v, eps)

        numerador += np.sum(np.log2(1 + ganho ** 2 * var_f / (var_v + sigma_n2)))  # informação que chega da distorcida
        denominador += np.sum(np.log2(1 + var_f / sigma_n2))                       # informação da própria referência
    return numerador / denominador, numerador                            # (VIF, IFC em bits)

print(f'{"Referência x referência":25s} VIF = {vif_ifc(ref, ref)[0]:.4f}')
for nome, g in distorcidas.items():
    vif, ifc = vif_ifc(ref, g)
    print(f'{nome:25s} VIF = {vif:.4f} | IFC = {ifc:12,.0f} bits')

## Métrica perceptual com Rede Neural (encoder)

Usamos a parte convolucional da **VGG16** pré-treinada no ImageNet como *encoder*: as duas imagens são transformadas em **mapas de características** e comparamos esses mapas, e não os pixels. É a ideia por trás da métrica **LPIPS**.

Quanto **menor** a distância, mais parecidas as imagens são para a rede.

In [ ]:
import torch
import torchvision.models as TModels

pesos = TModels.VGG16_Weights.DEFAULT
encoder = TModels.vgg16(weights=pesos).features.eval()                  # só as camadas convolucionais (encoder)
camadas = {3: 'relu1_2', 8: 'relu2_2', 15: 'relu3_3', 22: 'relu4_3'}  # saídas usadas na comparação
media_imnet = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
desvio_imnet = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def para_tensor(img):
    x = torch.from_numpy(np.stack([img] * 3, axis=0)).float()[None] / 255  # cinza -> 3 canais, formato (1,3,H,W)
    return (x - media_imnet) / desvio_imnet                             # normalização usada no treino da VGG

def caracteristicas(img):
    x, saidas = para_tensor(img), []
    with torch.no_grad():
        for i, camada in enumerate(encoder[:max(camadas) + 1]):
            x = camada(x)
            if i in camadas:
                saidas.append(x / (x.norm(dim=1, keepdim=True) + 1e-10))  # normaliza cada posição entre os canais
    return saidas

def distancia_perceptual(f, g):
    return sum(((a - b) ** 2).sum(dim=1).mean().item() for a, b in zip(caracteristicas(f), caracteristicas(g)))

for nome, g in distorcidas.items():
    print(f'{nome:25s} Distância perceptual (VGG16) = {distancia_perceptual(ref, g):.4f}')

# Métricas *No-Reference* (NR)

Avaliam a imagem **sozinha**, sem a referência. BRISQUE e NIQE usam estatísticas de cenas naturais (NSS): distorções alteram a distribuição dos coeficientes locais normalizados (MSCN) da imagem.

- **BRISQUE**: um modelo (SVR) treinado com notas subjetivas mapeia essas estatísticas em uma nota.
- **NIQE**: mede a distância entre as estatísticas da imagem e um modelo de imagens "naturais" sem distorção. Não usa notas subjetivas no treino.

Nas duas, quanto **menor**, melhor. Usamos a biblioteca `pyiqa`, que traz os modelos prontos.

In [ ]:
!pip install -q pyiqa

In [ ]:
import pyiqa

brisque = pyiqa.create_metric('brisque', device='cpu')
niqe = pyiqa.create_metric('niqe', device='cpu')

def para_pyiqa(img):
    return torch.from_numpy(img).float()[None, None] / 255              # formato (1,1,H,W) com valores em [0, 1]

for nome, g in {'Referência': ref, **distorcidas}.items():
    print(f'{nome:25s} BRISQUE = {brisque(para_pyiqa(g)).item():6.2f} | NIQE = {niqe(para_pyiqa(g)).item():5.2f}')

## Bônus: nitidez pela variância do Laplaciano

Uma medida NR simples de **borramento**: imagens nítidas têm muitas bordas, logo um Laplaciano com alta variância. Quanto **maior**, mais nítida (mas o ruído também aumenta o valor).

In [ ]:
def nitidez(img):
    return cv2.Laplacian(img.astype(np.float64), cv2.CV_64F).var()

for nome, g in {'Referência': ref, **distorcidas}.items():
    print(f'{nome:25s} Variância do Laplaciano = {nitidez(g):10.1f}')

# Resumo: todas as métricas lado a lado

↑ = quanto maior, melhor | ↓ = quanto menor, melhor

In [ ]:
linhas = []
for nome, g in distorcidas.items():
    vif, ifc = vif_ifc(ref, g)
    linhas.append({'Distorção': nome,
                   'MAE': mae(ref, g), 'MSE': mse(ref, g), 'PSNR (dB)': psnr(ref, g),
                   'SSIM': ssim(ref, g)[0], 'VIF': vif, 'IFC': ifc,
                   'VGG16': distancia_perceptual(ref, g),
                   'BRISQUE': brisque(para_pyiqa(g)).item(), 'NIQE': niqe(para_pyiqa(g)).item()})

pd.DataFrame(linhas).set_index('Distorção').round(3)

# Referências

- WANG, Z.; BOVIK, A. C.; SHEIKH, H. R.; SIMONCELLI, E. P. Image quality assessment: from error visibility to structural similarity. *IEEE Transactions on Image Processing*, v. 13, n. 4, 2004. (SSIM)
- SHEIKH, H. R.; BOVIK, A. C. Image information and visual quality. *IEEE Transactions on Image Processing*, v. 15, n. 2, 2006. (VIF)
- SHEIKH, H. R.; BOVIK, A. C.; DE VECIANA, G. An information fidelity criterion for image quality assessment using natural scene statistics. *IEEE Transactions on Image Processing*, v. 14, n. 12, 2005. (IFC)
- MITTAL, A.; MOORTHY, A. K.; BOVIK, A. C. No-reference image quality assessment in the spatial domain. *IEEE Transactions on Image Processing*, v. 21, n. 12, 2012. (BRISQUE)
- MITTAL, A.; SOUNDARARAJAN, R.; BOVIK, A. C. Making a "completely blind" image quality analyzer. *IEEE Signal Processing Letters*, v. 20, n. 3, 2013. (NIQE)
- ZHANG, R. et al. The unreasonable effectiveness of deep features as a perceptual metric. *CVPR*, 2018. (LPIPS)
- PHILLIPS, J. B.; ELIASSON, H. *Camera image quality benchmarking*. John Wiley & Sons, 2018.
- Documentação: scikit-image (https://scikit-image.org/docs/stable/api/skimage.metrics.html) e pyiqa (https://github.com/chaofengc/IQA-PyTorch)